git files

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, DateType, DoubleType, LongType

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

import mlflow
import mlflow.spark

import pandas as pd
import subprocess
import datetime as dt
import math
import random

In [0]:
dbutils.widgets.text("source_path", "")
dbutils.widgets.dropdown("ingest_mode", "csv", ["csv", "yfinance_btcusd", "synthetic"], "Ingest mode")
dbutils.widgets.text("yf_period", "1y")
dbutils.widgets.text("yf_start", "")
dbutils.widgets.text("yf_end", "")
dbutils.widgets.text("mlflow_experiment", "/Shared/bitcoin_lr_linear_regression")
dbutils.widgets.text("predictions_table", "")

source_path = dbutils.widgets.get("source_path").strip()
ingest_mode = dbutils.widgets.get("ingest_mode").strip()
yf_period = dbutils.widgets.get("yf_period").strip()
yf_start = dbutils.widgets.get("yf_start").strip()
yf_end = dbutils.widgets.get("yf_end").strip()
experiment_path = dbutils.widgets.get("mlflow_experiment").strip()
predictions_table = dbutils.widgets.get("predictions_table").strip()

def _dbfs_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def _install_yfinance_if_needed():
    try:
        import yfinance as yf
        return yf
    except Exception:
        %pip install yfinance
        import yfinance as yf
        return yf

def _parse_date(s: str):
    if not s:
        return None
    return dt.datetime.strptime(s, "%Y-%m-%d").date()

def _read_yfinance_btcusd():
    yf = _install_yfinance_if_needed()
    today = dt.date(2026, 3, 17)
    one_year_ago = today - dt.timedelta(days=365)
    pdf = yf.download(
        "BTC-USD",
        start=one_year_ago.strftime("%Y-%m-%d"),
        end=today.strftime("%Y-%m-%d"),
        interval="1d",
        auto_adjust=False,
        progress=False
    )
    if pdf is None or len(pdf) == 0:
        raise ValueError("No rows returned from yfinance for BTC-USD")
    pdf = pdf.reset_index()
    rename = {"Date": "date", "Open": "open", "High": "high", "Low": "low", "Close": "close", "Volume": "volume"}
    pdf = pdf.rename(columns=rename)
    pdf["date"] = pd.to_datetime(pdf["date"]).dt.date
    for c in ["open", "high", "low", "close", "volume"]:
        pdf[c] = pd.to_numeric(pdf[c], errors="coerce")
    pdf = pdf.dropna(subset=["date", "close", "volume"])
    return spark.createDataFrame(pdf[["date", "open", "high", "low", "close", "volume"]])

def _normalize_columns(df):
    cols = {c.lower(): c for c in df.columns}
    rename_map = {}
    for canonical, candidates in {
        "date": ["date", "timestamp", "time"],
        "open": ["open"],
        "high": ["high"],
        "low": ["low"],
        "close": ["close", "adj close", "adj_close", "adjclose"],
        "volume": ["volume", "vol"]
    }.items():
        for cand in candidates:
            if cand in cols:
                rename_map[cols[cand]] = canonical
                break

    for old, new in rename_map.items():
        df = df.withColumnRenamed(old, new)
    return df

raw = None
if ingest_mode == "yfinance_btcusd":
    try:
        raw = _read_yfinance_btcusd()
    except Exception:
        raw = None

if raw is None and ingest_mode in ["csv", "yfinance_btcusd"] and source_path and _dbfs_exists(source_path):
    raw = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(source_path)
    )
    raw = _normalize_columns(raw)

if raw is None:
    random.seed(42)
    start = dt.date(2020, 1, 1)
    days = (dt.date(2026, 3, 17) - start).days + 1
    price = 10000.0
    rows = []
    for i in range(days):
        date = start + dt.timedelta(days=i)
        drift = 0.0004
        shock = random.gauss(0.0, 0.02)
        ret = drift + shock
        close = price * math.exp(ret)
        high = max(price, close) * (1.0 + abs(random.gauss(0.0, 0.01)))
        low = min(price, close) * (1.0 - abs(random.gauss(0.0, 0.01)))
        open_ = price
        volume = int(1_000_000 + abs(random.gauss(0, 200_000)))
        rows.append((date, float(open_), float(high), float(low), float(close), int(volume)))
        price = close

    schema = StructType([
        StructField("date", DateType(), False),
        StructField("open", DoubleType(), False),
        StructField("high", DoubleType(), False),
        StructField("low", DoubleType(), False),
        StructField("close", DoubleType(), False),
        StructField("volume", LongType(), False)
    ])
    raw = spark.createDataFrame(rows, schema=schema)

raw = (
    raw
    .withColumn("date", F.to_date(F.col("date")))
    .withColumn("close", F.col("close").cast("double"))
    .withColumn("volume", F.col("volume").cast("double"))
    .filter(F.col("date").isNotNull())
    .filter(F.col("close").isNotNull())
    .filter(F.col("volume").isNotNull())
    .filter(F.col("date") <= F.lit(dt.date(2026, 3, 17)))
)

raw.orderBy(F.desc("date")).display()

In [0]:
raw.write.mode("overwrite").saveAsTable("crypto_ml.cryptobronze.btc_raw")

In [0]:
display(
    spark.table("crypto_ml.cryptobronze.btc_raw")
    .orderBy("date", ascending=False)
)

In [0]:
w_order = Window.orderBy("date")
w_ma7 = w_order.rowsBetween(-6, 0)
w_rsi14 = w_order.rowsBetween(-13, 0)

df = (
    raw.orderBy("date")
    .withColumn("prev_close", F.lag("close", 1).over(w_order))
    .withColumn("return_1d", (F.col("close") - F.col("prev_close")) / F.col("prev_close"))
    .withColumn("ma7_close", F.avg("close").over(w_ma7))
)

df = (
    df
    .withColumn("delta", F.col("close") - F.col("prev_close"))
    .withColumn("gain", F.when(F.col("delta") > 0, F.col("delta")).otherwise(F.lit(0.0)))
    .withColumn("loss", F.when(F.col("delta") < 0, -F.col("delta")).otherwise(F.lit(0.0)))
    .withColumn("avg_gain_14", F.avg("gain").over(w_rsi14))
    .withColumn("avg_loss_14", F.avg("loss").over(w_rsi14))
    .withColumn(
        "rsi14",
        F.when(F.col("avg_loss_14") == 0, F.lit(100.0))
        .otherwise(100.0 - (100.0 / (1.0 + (F.col("avg_gain_14") / F.col("avg_loss_14")))))
    )
)

df = df.withColumn("label_return_1d_ahead", F.lead("return_1d", 1).over(w_order))

dataset = (
    df
    .select("date", "close", "volume", "return_1d", "ma7_close", "rsi14", "label_return_1d_ahead")
    .filter(F.col("return_1d").isNotNull())
    .filter(F.col("ma7_close").isNotNull())
    .filter(F.col("rsi14").isNotNull())
    .filter(F.col("label_return_1d_ahead").isNotNull())
)

#dataset.orderBy("date").display()

In [0]:
dataset.write.mode("overwrite").saveAsTable("crypto_ml.cryptosilver.btc_features")


In [0]:
display(
    spark.table("crypto_ml.cryptosilver.btc_features")
    .orderBy("date", ascending=False)
)

In [0]:
n = dataset.count()
train_n = int(n * 0.8)

dataset_idx = dataset.withColumn("row_num", F.row_number().over(w_order))
train_df = dataset_idx.filter(F.col("row_num") <= F.lit(train_n)).drop("row_num")
test_df = dataset_idx.filter(F.col("row_num") > F.lit(train_n)).drop("row_num")

train_df.select(F.min("date"), F.max("date"), F.count("*").alias("rows")).display()
test_df.select(F.min("date"), F.max("date"), F.count("*").alias("rows")).display()

In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

label_col = "label_return_1d_ahead"
feature_cols = ["ma7_close", "rsi14", "volume", "return_1d"]

# Convert Spark DataFrames to pandas
train_pdf = train_df.select(["date", "close"] + feature_cols + [label_col]).toPandas()
test_pdf  = test_df.select(["date", "close"] + feature_cols + [label_col]).toPandas()

# Keep only needed columns and coerce numeric types
for c in feature_cols + [label_col, "close"]:
    train_pdf[c] = pd.to_numeric(train_pdf[c], errors="coerce")
    test_pdf[c] = pd.to_numeric(test_pdf[c], errors="coerce")

# Drop rows with missing label
train_pdf = train_pdf.dropna(subset=[label_col]).copy()
test_pdf = test_pdf.dropna(subset=[label_col]).copy()

X_train = train_pdf[feature_cols]
y_train = train_pdf[label_col]
X_test = test_pdf[feature_cols]
y_test = test_pdf[label_col]

model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("lr", LinearRegression())
])

mlflow.set_registry_uri("databricks")

with mlflow.start_run(run_name="bitcoin_lr_sklearn"):
    mlflow.log_param("framework", "sklearn")
    mlflow.log_param("label", label_col)
    mlflow.log_param("features", ",".join(feature_cols))
    mlflow.log_param("train_rows", int(len(train_pdf)))
    mlflow.log_param("test_rows", int(len(test_pdf)))

    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
    r2 = float(r2_score(y_test, pred))

    pred_up = (pred >= 0).astype(int)
    label_up = (y_test.to_numpy() >= 0).astype(int)
    direction_acc = float((pred_up == label_up).mean())

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    mlflow.log_metric("direction_accuracy", direction_acc)

    mlflow.sklearn.log_model(model, artifact_path="model")

# Build predictions table for display
preds_pdf = test_pdf[["date", "close"] + feature_cols + [label_col]].copy()
preds_pdf["prediction"] = pred
preds_pdf = preds_pdf.sort_values("date", ascending=False)

#display(preds_pdf)

In [0]:
gold_df = spark.createDataFrame(preds_pdf)
gold_df.write.mode("overwrite").saveAsTable("crypto_ml.cryptogold.btc_predictions")

In [0]:
display(
    spark.table("crypto_ml.cryptogold.btc_predictions")
    .orderBy("date", ascending=False)
)

## Optional: save predictions to a table

Set `predictions_table` widget to a table name to persist predictions.

In [0]:
if predictions_table:
    (
        preds
        .select(
            "date",
            "close",
            "volume",
            "ma7_close",
            "rsi14",
            "return_1d",
            F.col(label_col).alias("label"),
            "prediction"
        )
        .write
        .mode("overwrite")
        .saveAsTable(predictions_table)
    )
    spark.table(predictions_table).orderBy(F.desc("date")).display()